In [1]:
print("hello")

hello


In [10]:
### Basic deep agent

import os
from dotenv import load_dotenv
load_dotenv()

os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")
os.environ["TAVILY_API_KEY"]=os.getenv("TAVILY_API_KEY")

In [11]:
import os
from deepagents import create_deep_agent
from deepagents.backends import StateBackend

In [21]:
# -----------------------------------------------------------------------------
# 1. Create the agent — these two are equivalent
# -----------------------------------------------------------------------------
agent = create_deep_agent(model="google_genai:gemini-3.5-flash")

# Under the hood this is what `agent` is doing — explicit StateBackend:
agent2 = create_deep_agent(
    model="google_genai:gemini-3.5-flash",
    backend=StateBackend(),
)

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.
Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


In [22]:
# -----------------------------------------------------------------------------
# 2. Invoke the agent and ask it to WRITE a file
#    (StateBackend keeps that file inside LangGraph state)
# -----------------------------------------------------------------------------
result = agent2.invoke({
    "messages": [{
        "role": "user",
        "content": (
            "Create a file at /notes/todo.txt with exactly this content:\n"
            "1. Record video\n2. Edit video\n3. Upload video\n"
            "Then tell me you've done it."
        )
    }]
})
# The agent's final natural-language reply
print("\n--- Agent reply -------------------------------------------------")
print(result["messages"][-1].content)


--- Agent reply -------------------------------------------------
[{'type': 'text', 'text': 'I have created the file at `/notes/todo.txt` with the requested content.', 'extras': {'signature': 'Es8DCswDAQw51seX+IWyKB3Ld3GspmJ0EYHyHG2AMXYSG1+uaiY0w5dPQ2/rRO//2OQ3GO5yJflQEwCq1T6AozO6Fgjod+Eh/YbfbeuESkpNMLJZgwv4baM5hHPoK/4vuQt7mMloHR/wbRSE9u4JmOH67YeuEJ8nge/yDhW8xbLvqgxVV31Hfcm+e+c2wqe+92+FOK4m+NzuSOf9TUgp/PIiCryLG6+VYIgvl2y7AhxNxfzxiEECGITaQNZAWp364EMe1ExmClAmCoL586bjXgKh3Ia4hszWzQJf6YZv3LHIR6Gwf0xRGIHo1gr1DlndVzCntjCo/RkRQWAVXYgWvnw3woYsglo7WvUz+DNEUXVtkR99LGwrNVE1KXGU0VszJUpzO8udWOxA2CjpwIRPYpddCjKwAOZz23n0pcszErGUXkp4EMZgfnSBkRY+vNNqRk2HIhz5D5OothogTGvneiAD+b5Mq1fzNugGFLXRuEUPz1VkNuVxEpJN0FS3BwkfJTB400ox9Tezv08zihQMJFu0BYhQcVn7Hb9yI3y4VryR++iI8SfwXErb+tS77Y3W0Ww1lQDDmAIaky3jxI8NrhtHIUNIUCoTVX52dZzKFXzikw=='}}]


In [23]:
result

{'messages': [HumanMessage(content="Create a file at /notes/todo.txt with exactly this content:\n1. Record video\n2. Edit video\n3. Upload video\nThen tell me you've done it.", additional_kwargs={}, response_metadata={}, id='349bb433-9667-491c-960d-7f55755f98f4'),
  AIMessage(content=[], additional_kwargs={'function_call': {'name': 'write_file', 'arguments': '{"content": "1. Record video\\n2. Edit video\\n3. Upload video\\n", "file_path": "/notes/todo.txt"}'}, '__gemini_function_call_thought_signatures__': {'tqra1zuy': 'EoYHCoMHAQw51scAQNXXLR1TZEZFEBVG4t9D9OLo10Xwgs9hV+7xQSnAFi4L5W+oQIgxqhfG47ez+KGbTz2g/FJa0ghnq09uDkSiv2vJ69fZvDGfEOyA7G+r4Gn8jqVyx8YeQefpJNso7AqjeSQxVTjXOGwqHRjR1iNBqOHdLvyERmvnyZh9Etd/sSJwAUAH/Y0BjCK/mj4UzYJ8s06jRY201bQXrBbYYuue76srHxbLV14VXgqzUfapVErfL7h0rvYg327JdioOcaStk+TK1OprpXroCUGmPjBb7DHEa7jImhpDsR+Uj8DQKr2lfBa6eMvfSeLf2Gm3gM/zD0yfdulUd55Ch0jC9G4Ou4Tg3kHcUtrXroCGo4xI14e37YTN12WO7Unf+LEcY6DjHZzdS41XaklNJouhj2J9qvjnPEforaaGw+aFF6tI83D+IgmG4prYW1so3uRBZ0FfYILaXFEGuK

In [24]:
# -----------------------------------------------------------------------------
# 3. CHECK the backend is working
#    With StateBackend, written files appear under result["files"]
# -----------------------------------------------------------------------------
print("\n--- Backend check -----------------------------------------------")
files = result.get("files", {})

if files:
    print(f"✅ StateBackend is working — {len(files)} file(s) in state:")
    for path, content in files.items():
        print(f"\n📄 {path}\n{'-' * 40}\n{content}")
else:
    print("⚠️  No files found in state. Either the agent didn't write a file, "
          "or the backend isn't wired up correctly.")


--- Backend check -----------------------------------------------
✅ StateBackend is working — 1 file(s) in state:

📄 /notes/todo.txt
----------------------------------------
{'content': '1. Record video\n2. Edit video\n3. Upload video\n', 'encoding': 'utf-8', 'created_at': '2026-06-07T08:39:50.394014+00:00', 'modified_at': '2026-06-07T08:39:50.394014+00:00'}


In [25]:
# -----------------------------------------------------------------------------
# 4. Prove persistence WITHIN the same thread:
#    feed the returned state back in and ask it to READ the file
# -----------------------------------------------------------------------------
followup = agent2.invoke({
    # carry forward prior messages + the files state
    "messages": result["messages"] + [{
        "role": "user",
        "content": "Read /notes/todo.txt back to me ."
    }],
    "files": result.get("files", {}),   # <-- pass the virtual filesystem along
})

print("\n--- Read-back (same thread) -------------------------------------")
print(followup["messages"][-1].content)


--- Read-back (same thread) -------------------------------------
[{'type': 'text', 'text': 'Here is the content of `/notes/todo.txt`:\n\n1. Record video\n2. Edit video\n3. Upload video', 'extras': {'signature': 'ErQBCrEBAQw51sd1ViTkFCsDk2lpYHTUTr5kpKKlMoCsz2tHrpI+1PHkx4kGuvY2g8w0m6sr9xzoMPs3C+UbLc7DPuRk7SAHxBJNIn5BaFtvVgvQDskMpPfoJV+cvkCsfBUzT9gNWMmKMkO6HM9ycJWlnjjPPjUtFjVmU+Ig7VLvoltImZrDZ84lIyhXOR+bNe6vv0QsITtMW04OOR1NCh+H2lwD50hBJxLBRwFslRnBLlOWE2Lp'}}]


In [26]:
#File System Backend
# -----------------------------------------------------------------------------
# 1. Create the agent with a real-disk backend
#    root_dir="." -> files land relative to your current working directory
#    virtual_mode=True -> agent uses virtual paths like /notes/todo.txt,
#                         mapped onto root_dir
# -----------------------------------------------------------------------------
from deepagents.backends import FilesystemBackend
ROOT = "."

agent=create_deep_agent(model="google_genai:gemini-3.5-flash",backend=FilesystemBackend(root_dir=ROOT,virtual_mode=True))
print(f"✅ Agent created with FilesystemBackend(root_dir={ROOT!r}).")
print("   Files written by the agent will appear on your ACTUAL disk.")

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


✅ Agent created with FilesystemBackend(root_dir='.').
   Files written by the agent will appear on your ACTUAL disk.


In [27]:
# -----------------------------------------------------------------------------
# 2. Invoke the agent and ask it to WRITE a file
# -----------------------------------------------------------------------------
result = agent.invoke({
    "messages": [{
        "role": "user",
        "content": (
            "Create a file at /notes/todo.txt with exactly this content:\n"
            "1. Record video\n2. Edit video\n3. Upload video\n"
            "Then tell me you've done it."
        )
    }]
})

print("\n--- Agent reply -------------------------------------------------")
print(result["messages"][-1].content)


--- Agent reply -------------------------------------------------
[{'type': 'text', 'text': 'I have created the file at `/notes/todo.txt` with exactly the requested content.', 'extras': {'signature': 'EqwKCqkKAQw51sck59B6kJ+ZsD6lUFAdjxXji4meDfFoEK24WPDByZhhPLrUyd6S5ykyRF5eyEA7tK8wFzLUmOdRN2Vhy1MhddM21oD0t5LhSOs4tCayRXHJ++Mqml5XZ/Ybr8YL1qWp6XJfbeAi/Gam6XV2MPoLXN6edMtoqxvmKmzqtr5lB+cIuPMVC3ypgp5RV6bvF1a75OeTnr4x38V1ShgTZAsBBw3wbIzZH/FV+eMOAMe+6QyDHmtXum0hd7vLr5GAxUOHdlJrZOd4ssAa0KAmfPeGgY0FW+cZ1VgB1Bj2h4eJr6f++Syb8X/3PrpL9eDJLBDCDyfGKtRYcZHlZbFTyvaYVf8qGFlYI3Bi9I/lKNQLkzSA+H3d4vt0Xi6BMhR/roZ00Bh1g05SxO7MZOSAu7kah7WIT8ORBra1vLF8kqSHwPZRNC0F3KgyJZyj3BaNukXusyUVk8MHehdEAcFZwftltfeYTIbn7GrW3+ChmPI5us/hqENGR1KJrX8UWZ/J8UWegJk0ZTignu9Ejni03zvkVihVx4G0MlhH6TEGT6xnQRNJ81ddZBxQ53Mw/1jNDOWObpTE4brisCnEjZ0raXROnPLFyCJDMo/810YnmC3UCh0OXNjYiZxtXDviOSqmUhOjlVnCRYu1YWMYdUueaeNgUttZPSpXPmPWSCThfDrpm8E+5aRE14tsA7GftY2hD0TskrWoCTz7zDH2HYM+oOO/1diCiCm2t8Eu3z9q9wh6NZbgo4twpBNYvctkB/rlDqsiO4ECs4Vw1KrtgAz7hz

In [28]:
# -----------------------------------------------------------------------------
# 3. CHECK the backend is working — look on the REAL disk
#    With virtual_mode=True, /notes/todo.txt maps to ./notes/todo.txt
# -----------------------------------------------------------------------------
from pathlib import Path
print("\n--- Backend check (real filesystem) -----------------------------")
disk_path = Path(ROOT) / "notes" / "todo.txt"

if disk_path.exists():
    print(f"✅ FilesystemBackend is working — file exists on disk:")
    print(f"📄 {disk_path.resolve()}\n{'-' * 40}")
    print(disk_path.read_text())
else:
    print(f"⚠️  Expected file not found at {disk_path.resolve()}")
    print("    The agent may not have called the write tool, or the path "
          "mapping differs.")


--- Backend check (real filesystem) -----------------------------
✅ FilesystemBackend is working — file exists on disk:
📄 E:\deep_agents\deep_agent_overview\notes\todo.txt
----------------------------------------
1. Record video
2. Edit video
3. Upload video


In [29]:
# -----------------------------------------------------------------------------
# 4. Prove persistence ACROSS sessions:
#    Unlike StateBackend, this file survives even after Python exits.
#    A brand-new agent (fresh state) can read it back from disk.
# -----------------------------------------------------------------------------
fresh_agent = create_deep_agent(
    model="google_genai:gemini-3.5-flash",
    backend=FilesystemBackend(root_dir=ROOT, virtual_mode=True),
)

followup = fresh_agent.invoke({
    "messages": [{
        "role": "user",
        "content": "Read /notes/todo.txt back to me verbatim."
    }]
    # NOTE: no `files` state passed in — the file is read straight from disk
})

print("\n--- Read-back with a FRESH agent (proves disk persistence) ------")
print(followup["messages"][-1].content)

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.



--- Read-back with a FRESH agent (proves disk persistence) ------
[{'type': 'text', 'text': '1. Record video\n2. Edit video\n3. Upload video', 'extras': {'signature': 'ErMBCrABAQw51sdWbk/eVOjqQVAiv/HsIju281Cd+ffWwTukivyOEAXBeGVpiag93M0bpgoSYB58mnV8VoO5maecuhbiNIgWm5A+w0f1qSaaTF3RAmyi4t79QwGoiy2fLTzQJ0W3R8sHM+0khriW990HOl30MGSQGPdU5DEss6Atu0dhyu5zOaBP+pXHt+Eo23Y/14pXCneAU9MxdIxxsxwBnIX42LH2Y/eiGB6fhbPweCXrKLs='}}]


Deep Agent — StoreBackend verification
Creates a deep agent backed by a LangGraph store, invokes it to write a file on one thread, then proves the backend works by reading that file back on a DIFFERENT thread — something StateBackend cannot do. """

In [30]:
from langgraph.store.memory import InMemoryStore
from deepagents.backends import StoreBackend
store = InMemoryStore()

agent = create_deep_agent(
    model="google_genai:gemini-3.5-flash",
    backend=StoreBackend(
        # Local dev: static namespace. No deployment runtime needed.
        # In a LangSmith Deployment you'd use the user-identity version.
        namespace=lambda rt: ("demo-user",),
    ),
    store=store,
)

print("✅ Agent created with StoreBackend + static namespace.")

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


✅ Agent created with StoreBackend + static namespace.


In [31]:
import os
import uuid
# -----------------------------------------------------------------------------
# 2. THREAD 1 — write a file
# -----------------------------------------------------------------------------
thread_1 = {"configurable": {"thread_id": str(uuid.uuid4())}}

result = agent.invoke(
    {
        "messages": [{
            "role": "user",
            "content": (
                "Create a file at /notes/todo.txt with exactly this content:\n"
                "1. Record video\n2. Edit video\n3. Upload video\n"
                "Then tell me you've done it."
            )
        }]
    },
    config=thread_1,
)

print("\n--- Agent reply (thread 1) --------------------------------------")
print(result["messages"][-1].content)


--- Agent reply (thread 1) --------------------------------------
[{'type': 'text', 'text': 'I have created the file at `/notes/todo.txt` with the requested content.', 'extras': {'signature': 'EsAECr0EAQw51scaL+PpCyiaq3gI01uPChKroU06tnQDrmnurYZuu/phwRCgW5W8P8zcvHT0RzZy5+Zu8YMNMuVlMdnWPqXn0vSZGTHlk3atIt/IiZyWBFWaiY97jbfs+ze1QimkJH7AWinIhYdWrpXqGwt0wed5pnXk3KRQ2ngJaBW/CAnzcKNyMItwxpztJIjedo29+x7glxkdVnwUWTj6FEWhhRbZG4e0fvat9QKF7h0sXsgFQ/mmF+UbMh92sXR0+7LQgkPz0D7yqb4bftibm9zaZdIho8DuBmk553GdaosH3kMqMIgaLCSmgXOCBJXZGetAIbK5VDO7xslpKsKcW+QGlb+q8EbNQ5bjO7f+JT2bhzNvxPCMtyUd5dpCeDO+CC9LCQOoeDXWjq4JMuia5sH/Bj3xGG39kMJYprY9Zdm92Z2F3cYxDyS+wqQ8pW+JAhCQqzmRF+EawvO8v8yLfGVeSZXTDk1sPRN8AaOk2HBQ7aTBxrxxNH/QGUa3J71DAnk7VtONftq9PK7C7r8xMxtvFGYbziw8rtUN+1QGN8xURzmOzfWq0vDRNQtUANUqdHkmF4c7yYk0gMefS5swPVTZMGqmvg2KMTiGqBEElRONzkU7YzdrnfsdvLVn8WMY6t39Jk4YVr0GeLenS/68csaN7iLmyzANzcrjDLkFwKi/SvNog2/xaO+Gn613HAnAw0Rdo1ysYH3T0c7nK02stS+MfexGPs/ZkFlFmUvkez1ivvQRk7PYpLF+exap9nbm'}}]


In [32]:
# --- Thread 2: read back on a DIFFERENT thread ---------------------------
thread_2 = {"configurable": {"thread_id": str(uuid.uuid4())}}
followup = agent.invoke(
    {"messages": [{
        "role": "user",
        "content": "Read /notes/todo.txt back to me verbatim."
    }]},
    config=thread_2,
)
print("\n--- Read-back on a different thread ---")
print(followup["messages"][-1].content)


--- Read-back on a different thread ---
[{'type': 'text', 'text': '1. Record video\n2. Edit video\n3. Upload video', 'extras': {'signature': 'EqsDCqgDAQw51sdMLVkX2be3zNgSerns6285AAlgzzZBa5KqjqQLSRUovekz6dEVokIKsvNg8T05HB43S0aspARouSWLkHtdRMvK5PVvTYpBM39mwi+OKK91CXLXa3YnIl4KusbBUNrxAGyPVQ1FSXUuaA58zH2n+CchgvCKsynx56V1ftD2ATXm6E7N49xDUFdoH/g+tShpwPXR17um/+gvxdNJu51NZWGvGaKa1EKddAdetxudMRoJGtGI8AaLEEsgL47NgLc4VCMYzsZ3XBD1V1Q+xIUa4MT4sKUmbWBjYV9i+/JahZMLXsuWhI2FZZXVw16snyjK8SjJUEir4wldnd5vEk61ZM+xVCESKcfD0qG7DfQi1pYu1mfvKO2KospzDTVfGD5IcYK0E7dYhni9c9TH81GQ8+iJ2YmfrAZPTiHPNbtK49AN7a+ZQLJEhtFa3uubb0CkVNAhR2BZ8Ewnkv49CZ+0vifwjTs9pynvtBckPegEtBonfDq7TlWpVyEKt8XBVIrwYOpGCbK0UwYnBANRkscfKmdPjXuMTmnTAJZrZj71werGlIGExw=='}}]


With StoreBackend + InMemoryStore, the file is not saved to disk at all. It lives in RAM, inside the InMemoryStore object, as an entry keyed under the namespace.

**Deep Agent Backends — Where Your Files Actually Live**
Every Deep Agent works with a virtual filesystem: its tools read and write files using paths like /notes/todo.txt. But those paths are an abstraction. The backend decides where that data physically lives — and that single choice changes everything about persistence, sharing, and durability.

The agent code stays identical across all three. Only the backend changes.

1. StateBackend (default)
Files live in the agent's LangGraph state — i.e. in RAM, tied to a single thread. They're available during the run via result["files"], and you must manually carry that state forward to keep using them.

Where: in-memory dict, inside one thread's state
Survives: within a single conversation/thread only
Gone when: the thread ends or the process exits
Use it for: ephemeral scratch space, temporary working files
2. FilesystemBackend
Files are written to your real disk, under root_dir. With virtual_mode=True, a virtual path like /notes/todo.txt maps to root_dir/notes/todo.txt. These are genuine files you can open in an editor or cat from a terminal.

Where: your actual filesystem, relative to root_dir
Survives: across process restarts — it's a real file
Use it for: when the agent should edit real project files
⚠️ Caution: grants the agent real read/write disk access — point it at a scratch directory, not anything important
3. StoreBackend
Files live in a LangGraph store, scoped by a namespace. This is the only backend whose files persist across threads/conversations — write on one thread, read back on another. With InMemoryStore this works within a single process; swap in a persistent store (e.g. Postgres-backed) for true cross-restart durability.

Where: inside the store object, under a namespace key
Survives: across threads sharing the same store
Gone when: the process exits (if using InMemoryStore) — use a persistent store to outlive restarts
Use it for: long-term memory, per-user file spaces
Quick comparison
Backend	Lives in	Cross-thread?	Survives restart?	Real file on disk?
StateBackend	LangGraph state	❌	❌	❌
FilesystemBackend	Your disk	✅	✅	✅
StoreBackend	A LangGraph store	✅	only w/ persistent store	❌
Mental model: the agent never knows the difference. Its tools always speak in file paths — the backend quietly translates every read/write into the right operation underneath. That's what lets you swap backends without touching a line of agent logic.